In [1]:
import pandas as pd
from pathlib import Path

In [2]:
CSV_PATH = Path("interoperability_evaluation_us_dicom.csv")

df = pd.read_csv(CSV_PATH)

# ------------------------------------------------------------
# NORMALIZE COLUMN NAMES
# ------------------------------------------------------------
# print(df.columns)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)
# print(df.columns)
TOTAL = len(df)
print(f"Total DICOM files analyzed : {TOTAL:,}")

Total DICOM files analyzed : 3,468


In [3]:
# print(df.columns)
print("AVAILABLE COLUMNS")
print("-" * 20)
for col in df.columns:
    print(col)

AVAILABLE COLUMNS
--------------------
filename
dataset
comparison_status
manufacturer
manufacturer_model
sop_class
transfer_syntax
photometric
n_frames
doppler_candidate
evidence
raw_array_shape
monai_shape_chw
acquisition_type
monai_ready
spp_validation
root_cause


In [4]:
def print_distribution(title, column, top_n=None):

    if column not in df.columns:
        print(f"\n[{title}]")
        print("-" * 80)
        print(f"Column not found: {column}")
        return

    print(f"\n[{title}]")
    print("-" * 80)

    counts = (
        df[column]
        .fillna("MISSING")
        .astype(str)
        .value_counts()
    )

    if top_n:
        counts = counts.head(top_n)

    for value, count in counts.items():

        pct = 100 * count / TOTAL

        print(f"{str(value)[:60]:60} : {count:6,} ({pct:6.2f}%)")

In [5]:
print_distribution("DATASET DISTRIBUTION", 'dataset')
print_distribution("MANUFACTURER DISTRIBUTION", 'manufacturer')
print_distribution("MANUFACTURER MODEL DISTRIBUTION", 'manufacturer_model')
print_distribution("DOPPLER ULTRASOUND DETECTION", 'doppler_candidate')
print_distribution("PHOTOMETRIC DISTRIBUTION", 'photometric')
print_distribution("SOP CLASS DISTRIBUTION", 'sop_class')
print_distribution("TRANSFER SYNTAX DISTRIBUTION", 'transfer_syntax')
print_distribution("ACQUISITION TYPE DISTRIBUTION", 'acquisition_type')


[DATASET DISTRIBUTION]
--------------------------------------------------------------------------------
TCIA_IDC_Prostate_MRI_US_Biopsy                              :  1,762 ( 50.81%)
UTA4                                                         :  1,132 ( 32.64%)
ReMIND                                                       :    320 (  9.23%)
UTA7                                                         :    198 (  5.71%)
UTA10                                                        :     27 (  0.78%)
Aliza                                                        :     22 (  0.63%)
AIIMS                                                        :      7 (  0.20%)

[MANUFACTURER DISTRIBUTION]
--------------------------------------------------------------------------------
Eigen                                                        :  1,762 ( 50.81%)
SIEMENS                                                      :  1,296 ( 37.37%)
PixelMed                                                     :   

In [6]:
# ============================================================
# MANUFACTURER -> MODEL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("MANUFACTURER -> MODEL DISTRIBUTION")
print("=" * 80)

# Clean columns
df["manufacturer"] = (
    df["manufacturer"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
)

df["manufacturer_model"] = (
    df["manufacturer_model"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# GROUPED SUMMARY
# ------------------------------------------------------------

grouped = (
    df.groupby(["manufacturer", "manufacturer_model"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["manufacturer", "count"],
        ascending=[True, False]
    )
)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

for manufacturer in grouped["manufacturer"].unique():
    print(f"[{manufacturer}]")
    subset = grouped[
        grouped["manufacturer"] == manufacturer
    ]
    for _, row in subset.iterrows():
        model = row["manufacturer_model"]
        count = row["count"]
        pct = 100 * count / len(df)
        print(
            f"{model[:50]:50} : "
            f"{count:6,} ({pct:6.2f}%)"
        )
    print("-" * 80)


MANUFACTURER -> MODEL DISTRIBUTION
[ACUSON]
UNKNOWN                                            :      2 (  0.06%)
--------------------------------------------------------------------------------
[Agfa]
CR 35                                              :      4 (  0.12%)
CR 25                                              :      2 (  0.06%)
CR 75                                              :      1 (  0.03%)
--------------------------------------------------------------------------------
[Eigen]
Artemis                                            :  1,762 ( 50.81%)
--------------------------------------------------------------------------------
[GE Healthcare]
LOGIQE10                                           :      1 (  0.03%)
LOGIQE9                                            :      1 (  0.03%)
--------------------------------------------------------------------------------
[GE Healthcare Austria GmbH & Co OG]
V830                                               :      1 (  0.03%)
---

In [7]:
# print_distribution("MANUFACTURER DISTRIBUTION", 'manufacturer')

In [8]:
# print_distribution("MANUFACTURER MODEL DISTRIBUTION", 'manufacturer_model')

In [9]:
# print_distribution("DOPPLER ULTRASOUND DETECTION", 'doppler_candidate')

In [10]:
# print_distribution("PHOTOMETRIC DISTRIBUTION", 'photometric')

In [11]:
# print_distribution("SOP CLASS DISTRIBUTION", 'sop_class')

In [12]:
# print_distribution("TRANSFER SYNTAX DISTRIBUTION", 'transfer_syntax')

In [13]:
# print_distribution("", 'evidence')

In [14]:
print_distribution("RAW ARRAY SHAPE DISTRIBUTION", 'raw_array_shape')


[RAW ARRAY SHAPE DISTRIBUTION]
--------------------------------------------------------------------------------
352x352                                                      :    784 ( 22.61%)
288x288                                                      :    512 ( 14.76%)
281x436x436                                                  :    477 ( 13.75%)
227x360x360                                                  :    213 (  6.14%)
227x352x352                                                  :    181 (  5.22%)
226x358x358                                                  :     89 (  2.57%)
227x378x378                                                  :     86 (  2.48%)
227x364x364                                                  :     77 (  2.22%)
280x440x440                                                  :     71 (  2.05%)
227x366x366                                                  :     68 (  1.96%)
227x356x356                                                  :     67 (  1.93%)
280x424

In [15]:
print_distribution("MONAI SHAPE DISTRIBUTION", 'monai_shape_chw')


[MONAI SHAPE DISTRIBUTION]
--------------------------------------------------------------------------------
(1, 352, 352)                                                :    784 ( 22.61%)
(1, 288, 288)                                                :    512 ( 14.76%)
(1, 281, 436, 436)                                           :    477 ( 13.75%)
(1, 227, 360, 360)                                           :    213 (  6.14%)
(1, 227, 352, 352)                                           :    181 (  5.22%)
(1, 226, 358, 358)                                           :     89 (  2.57%)
(1, 227, 378, 378)                                           :     86 (  2.48%)
(1, 227, 364, 364)                                           :     77 (  2.22%)
(1, 280, 440, 440)                                           :     71 (  2.05%)
(1, 227, 366, 366)                                           :     68 (  1.96%)
(1, 227, 356, 356)                                           :     67 (  1.93%)
(1, 280, 42

In [16]:
# print_distribution("ACQUISITION TYPE DISTRIBUTION", 'acquisition_type')

In [17]:
print_distribution("FAILURE ROOT CAUSE", 'root_cause')


[FAILURE ROOT CAUSE]
--------------------------------------------------------------------------------
Both tools read the file successfully. No issues detected.   :  3,468 (100.00%)


In [18]:
# # ============================================================
# # GENERIC DISTRIBUTION VISUALIZATION SCRIPT
# # ============================================================

# import pandas as pd
# import matplotlib.pyplot as plt


# # ------------------------------------------------------------
# # LOAD CSV
# # ------------------------------------------------------------

# df = pd.read_csv("interoperability_evaluation_us_dicom.csv")

# # Normalize column names
# df.columns = df.columns.str.strip().str.lower()


# # ============================================================
# # GENERIC VISUALIZATION FUNCTION
# # ============================================================

# def visualize_distribution(
#     df,
#     column,
#     title=None,
#     top_n=None,
#     figsize=(12, 6),
#     rotate_xticks=True,
#     save=False
# ):
#     """
#     Generic visualization for categorical distributions.

#     Parameters
#     ----------
#     df : pandas.DataFrame

#     column : str
#         Column name

#     title : str
#         Plot title

#     top_n : int
#         Keep only top N categories

#     figsize : tuple

#     rotate_xticks : bool

#     save : bool
#         Save figure as PNG
#     """

#     if column not in df.columns:
#         print(f"[SKIPPED] Column not found: {column}")
#         return

#     # --------------------------------------------------------
#     # VALUE COUNTS
#     # --------------------------------------------------------

#     counts = (
#         df[column]
#         .fillna("MISSING")
#         .astype(str)
#         .value_counts()
#     )

#     if top_n is not None:
#         counts = counts.head(top_n)

#     percentages = 100 * counts / counts.sum()

#     summary_df = pd.DataFrame({
#         "Category": counts.index,
#         "Count": counts.values,
#         "Percent": percentages.values
#     })

#     # --------------------------------------------------------
#     # PRINT TABLE
#     # --------------------------------------------------------

#     print("\n" + "=" * 80)

#     if title is None:
#         title = column.upper()

#     print(title)

#     print("=" * 80)

#     for _, row in summary_df.iterrows():

#         print(
#             f"{row['Category'][:60]:60} : "
#             f"{int(row['Count']):6,} "
#             f"({row['Percent']:6.2f}%)"
#         )

#     # --------------------------------------------------------
#     # VISUALIZATION
#     # --------------------------------------------------------

#     fig, ax = plt.subplots(figsize=figsize)

#     bars = ax.bar(
#         summary_df["Category"],
#         summary_df["Count"]
#     )

#     # Labels on bars
#     for bar, pct in zip(bars, summary_df["Percent"]):

#         height = bar.get_height()

#         ax.text(
#             bar.get_x() + bar.get_width() / 2,
#             height,
#             f"{pct:.1f}%",
#             ha='center',
#             va='bottom',
#             fontsize=9
#         )

#     # Axis labels
#     ax.set_xlabel(column.replace("_", " ").title())
#     ax.set_ylabel("Count")

#     # Title
#     ax.set_title(title)

#     # Rotate x labels
#     if rotate_xticks:
#         plt.xticks(rotation=45, ha="right")

#     plt.tight_layout()

#     # Save figure
#     if save:
#         filename = f"{column}_distribution.png"
#         plt.savefig(filename, dpi=300)
#         print(f"\nSaved: {filename}")

#     plt.show()


# # ============================================================
# # ALL DISTRIBUTIONS
# # ============================================================

# visualize_distribution(
#     df,
#     column="dataset",
#     title="DATASET DISTRIBUTION",
#     save=False
# )

# visualize_distribution(
#     df,
#     column="manufacturer",
#     title="MANUFACTURER DISTRIBUTION",
#     top_n=20,
#     save=False
# )

# visualize_distribution(
#     df,
#     column="manufacturer_model",
#     title="MANUFACTURER MODEL DISTRIBUTION",
#     top_n=20,
#     save=False
# )

# visualize_distribution(
#     df,
#     column="doppler_candidate",
#     title="DOPPLER ULTRASOUND DETECTION",
#     save=False
# )

# visualize_distribution(
#     df,
#     column="photometric",
#     title="PHOTOMETRIC INTERPRETATION DISTRIBUTION",
#     save=False
# )

# visualize_distribution(
#     df,
#     column="sop_class",
#     title="SOP CLASS DISTRIBUTION",
#     top_n=20,
#     save=False
# )

# visualize_distribution(
#     df,
#     column="transfer_syntax",
#     title="TRANSFER SYNTAX DISTRIBUTION",
#     save=False
# )

# visualize_distribution(
#     df,
#     column="acquisition_type",
#     title="ACQUISITION TYPE DISTRIBUTION",
#     save=False
# )